In [0]:
# Databricks Notebook: gold_publish.py
from pyspark.sql.functions import col
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Retrieve run_id from Databricks widget (set in bronze_ingest.py)
dbutils.widgets.text("run_id", "")
run_id = dbutils.widgets.get("run_id")

silver_path = "/Volumes/azure-medallion-university-chapters/sliver/university_chapters"
gold_path = "/Volumes/azure-medallion-university-chapters/gold/university_chapters"

# Read Silver data
silver_df = spark.read.parquet(silver_path)

# Gold = consumer-facing dataset (OK + WARNING only)
gold_df = silver_df.filter(col("dq_status").isin(["OK", "WARNING"]))

# Save Gold output
gold_df.write.mode("overwrite").parquet(gold_path)

# Log counts
rows_gold = gold_df.count()
rows_warned = gold_df.filter(col("dq_status") == "WARNING").count()
rows_ok = gold_df.filter(col("dq_status") == "OK").count()

print(f"Run {run_id}: Gold published. Total={rows_gold}, Warned={rows_warned}, OK={rows_ok}")